# ReelMela Aftermovie Engine — Colab notebook

Runs free on Google Colab (Python + MoviePy). No Firebase Storage, no Blaze: clips live on Cloudinary (free), clip lists live in Firestore (Spark).

**Ranking rule:** best verified views first. PRO clips are locked into the opening 15 seconds. Watch-time only breaks ties for the edit.

**Exports:** a 2–3 min aftermovie (16:9 + vertical) and a 30-sec collab reel.

**Payout reminder:** the aftermovie pays ₹0. Earnings run only on verified views of the one video each creator posts on their own account — ₹10,000 pool, cap ₹3,000 (30%), ₹100 minimum.

In [ ]:
!pip install -q moviepy==1.0.3
import os, json, requests
from moviepy.editor import VideoFileClip, concatenate_videoclips, CompositeVideoClip, TextClip, AudioFileClip, vfx
print('MoviePy ready')

## 1 · Config — fill these in

- `EVENT_NAME` must match the `event` field in Firestore `clips`.
- `CLIPS` is the list of clip URLs. Easiest way: open the ReelMela upload/event page, or export the `clips` collection from the Firebase console (Firestore → clips → Export) and paste the `videoUrl` values here.
- `VIEWS` maps each clip URL to the verified views of that creator's **one payout post** (from the `payoutPosts` collection on earn.html). PRO creators go in `PRO_URLS`.

In [ ]:
EVENT_NAME = 'Harda College Fest'
# Paste clip URLs here (from Firestore collection 'clips', field videoUrl):
CLIPS = [
    # 'https://res.cloudinary.com/gmppxw8j/video/upload/v1/reelmela-clips/abc123.mp4',
]
# Verified views per clip URL (from Firestore collection 'payoutPosts'):
VIEWS = {}
# Clip URLs belonging to PRO creators — locked into the opening 15 seconds:
PRO_URLS = []
# Royalty-free music: upload a file called music.mp3 to this Colab session first.
MUSIC_FILE = 'music.mp3'
OUT_DIR = '/content/reelmela_out'
os.makedirs(OUT_DIR, exist_ok=True)
print('Event:', EVENT_NAME, '| clips:', len(CLIPS))

## 2 · Dummy clips — test render with 20 clips

Run this cell to make 20 dummy clips (mixed orientations, shaky, low light) and test the render before any real fest. If the render survives these, it survives a real fest.

In [ ]:
from moviepy.editor import ColorClip
import random
random.seed(7)
dummy_dir = '/content/dummy_clips'
os.makedirs(dummy_dir, exist_ok=True)
DUMMY = []
for i in range(20):
    w, h = random.choice([(1280,720),(720,1280),(720,720)])
    color = (random.randint(40,255), random.randint(40,200), random.randint(40,255))
    clip = ColorClip(size=(w,h), color=color, duration=random.choice([15,20,25,30]))
    clip = clip.fx(vfx.colorx, random.uniform(0.5,1.0))  # some dim / low-light
    path = os.path.join(dummy_dir, 'dummy_%02d.mp4' % i)
    clip.write_videofile(path, fps=24, codec='libx264', audio=False, verbose=False, logger=None)
    DUMMY.append(path)
print('Made', len(DUMMY), 'dummy clips in', dummy_dir)
# To test-render with dummies, set: CLIPS = DUMMY  (and VIEWS = {p: random views})

## 3 · Rank clips — verified views first, PRO in the opening 15 seconds

In [ ]:
def rank_clips(urls, views, pro_urls):
    ranked = sorted(urls, key=lambda u: views.get(u, 0), reverse=True)
    pro = [u for u in ranked if u in set(pro_urls)]
    rest = [u for u in ranked if u not in set(pro_urls)]
    # Opening 15 seconds = PRO clips first, then the best of the rest.
    return pro + rest

ORDERED = rank_clips(CLIPS, VIEWS, PRO_URLS)
print('Ranked order (best first):')
for u in ORDERED:
    print(' ', VIEWS.get(u, 0), 'views —', u)

## 4 · Download clips from Cloudinary

In [ ]:
local = []
for i, url in enumerate(ORDERED):
    if url.startswith('/content/'):
        local.append(url); continue  # already-local dummy clips
    path = '/content/clip_%02d.mp4' % i
    r = requests.get(url, timeout=120)
    r.raise_for_status()
    open(path, 'wb').write(r.content)
    local.append(path)
print('Downloaded', len(local), 'clips')

## 5 · Stitch — crossfade transitions, title card, royalty-free music

In [ ]:
def load_clip(path, target=(1280,720)):
    c = VideoFileClip(path)
    # Trim every clip to at most 8 seconds for the aftermovie cut.
    c = c.subclip(0, min(8, c.duration))
    c = c.resize(height=target[1]) if c.h > c.w else c.resize(width=target[0])
    c = c.crop(width=target[0], height=target[1], x_center=c.w/2, y_center=c.h/2)
    return c.set_fps(24)

clips16 = [load_clip(p, (1280,720)) for p in local]
title = TextClip('ReelMela — ' + EVENT_NAME, fontsize=64, color='white', font='DejaVu-Sans-Bold', size=(1280,720)).set_duration(3).set_position('center')
parts = [title] + clips16
final16 = concatenate_videoclips(parts, method='compose', padding=-0.5)  # 0.5s crossfade
if os.path.exists(MUSIC_FILE):
    music = AudioFileClip(MUSIC_FILE).subclip(0, min(final16.duration, 180))
    final16 = final16.set_audio(music.volumex(0.5))
print('Aftermovie length:', round(final16.duration, 1), 'seconds')

## 6 · Export both cuts

In [ ]:
out16 = os.path.join(OUT_DIR, 'aftermovie_16x9.mp4')
final16.write_videofile(out16, fps=24, codec='libx264', audio_codec='aac', verbose=False, logger=None)
print('Wrote', out16)

# Vertical cut (9:16) — same clips, cropped tall.
clipsV = [load_clip(p, (720,1280)) for p in local]
finalV = concatenate_videoclips(clipsV, method='compose', padding=-0.5)
if os.path.exists(MUSIC_FILE):
    finalV = finalV.set_audio(AudioFileClip(MUSIC_FILE).subclip(0, min(finalV.duration, 180)).volumex(0.5))
outV = os.path.join(OUT_DIR, 'aftermovie_vertical.mp4')
finalV.write_videofile(outV, fps=24, codec='libx264', audio_codec='aac', verbose=False, logger=None)
print('Wrote', outV)

# 30-second collab reel — best clips first, 3 seconds each.
best = clips16[:10]
collab = concatenate_videoclips([c.subclip(0, 3) for c in best], method='compose')
outC = os.path.join(OUT_DIR, 'collab_reel_30s.mp4')
collab.write_videofile(outC, fps=24, codec='libx264', audio_codec='aac', verbose=False, logger=None)
print('Wrote', outC)
print('Done. Download the files from', OUT_DIR, 'and post the aftermovie from @reelmela — it pays ₹0, it is the showcase.')